# Menos datos: corte mas temprano, arranque y convergencia

**Pregunta.** Si el corte se adelanta, el plan se arma con menos historia y el test dura mas. Con menos
historia: cuanto empeora cada plan, si arranca peor (arranque en frio), cuanto tarda en dar su ganancia
(calentamiento) y si termina en el mismo nivel (convergencia). Los cortes tempranos dejan tests de hasta
162 dias, asi que tambien se ve que pasa despues de los 72 dias del notebook 05.

**Contenido.**
0. Que se simula.
1. Las simulaciones.
2. Semana a semana desde el corte.
3. Cuanto empeora con menos datos.
4. Se estabiliza con mas tiempo?
5. Lectura.

## 0. Que se simula

Los datos son 6 meses de lineas de picking. El **corte** separa los batches en dos: con los anteriores
(train) se arma el plan y se reproduce el deposito hasta el corte; los posteriores (test) se simulan. Se
prueban cinco cortes, con el 10, 20, 40, 50 y 60% de los batches en train; el de 60% es el de los notebooks
anteriores.

En cada corte se simulan, partiendo del deposito real tal como quedo en el corte:

| corrida | plan | como entra el plan | batches |
|---|---|---|---|
| hoy | ninguno (el deposito sigue como esta) | no entra: reposicion de los datos | FIFO |
| aleatorio | `random`: cada SKU en una ubicacion sorteada | regla `plan_aisle` | re-batching, $W = 4$ |
| greedy | `demand_greedy`: lo mas pedido en las ubicaciones mas cercanas al dock | regla `plan_aisle` | re-batching, $W = 4$ |
| aisle_fill | `aisle_fill`: lo mas pedido, llenando un pasillo por vez desde el dock | regla `plan_aisle` | re-batching, $W = 4$ |
| ganador | `merchant_aisle_fill`: los SKUs de cada merchant juntos, en un pasillo o dos | regla `plan_aisle` | re-batching, $W = 4$ |
| techo de greedy, aisle_fill y ganador | el mismo plan, aplicado entero antes de empezar | un SKU agotado vuelve a su ubicacion (`home`) | re-batching, $W = 4$ |

La regla `plan_aisle` muda un SKU solo cuando se agota, y lo lleva a su pasillo del plan; el detalle de la
simulacion esta en el notebook 05. Los planes solo toman de train la demanda de cada SKU: la lista de SKUs
y el merchant de cada uno salen del stock inicial, asi que no dependen del corte. El techo mide el plan en
si; la corrida desde el deposito real mide ademas cuanto tarda en entrar.

**Medidas.** Se cuenta el tiempo desde el corte: dia $d$ y semana $s = \lfloor d / 7 \rfloor$, solo semanas
completas. $M_s$ es la caminata media por batch de la semana $s$, en pulgadas. La **ventaja sobre el
aleatorio** de un plan es $100\,(1 - M^{\text{plan}} / M^{\text{aleatorio}})$, con los dos del mismo corte
y el mismo periodo: es cuanto menos camina que un plan sin datos que entra igual. Para comparar cortes con
tests de distinto largo se usa un **horizonte comun**: los primeros 72 dias despues del corte (el test mas
corto); el promedio es sobre esos 72 dias y el final, sobre los dias 58 a 71.

In [ ]:
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from abs_affinity_based_slotting.config import PROJECT_ROOT
from abs_affinity_based_slotting.experiments import (EvalSpec, InstanceSpec, SlottingSpec, Task, load_results,
                                                     register_results, run_tasks)

pd.set_option("display.width", 200)

# ---- configuracion (tabla de arriba)
DEPOSITOS = ["medium", "big"]
TRAIN = [0.1, 0.2, 0.4, 0.5, 0.6]            # fraccion de los batches antes del corte
REBATCHING = EvalSpec(window=4)              # deposito real, regla plan_aisle (stockout, aisle), 20 pasadas, S
TECHO = EvalSpec(window=4, start="plan", replenishment="home")
MODELOS = {"aleatorio": SlottingSpec("random", seed="0"), "greedy": SlottingSpec("demand_greedy"),
           "aisle_fill": SlottingSpec("aisle_fill"), "ganador": SlottingSpec("merchant_aisle_fill")}
CON_DATOS = ["greedy", "aisle_fill", "ganador"]
CORRIDAS = {                                 # corrida -> (plan, como se simula)
    "hoy": (SlottingSpec("vigente"), EvalSpec(window=1, replenishment="random")),
    **{m: (plan, REBATCHING) for m, plan in MODELOS.items()},
    **{f"techo {m}": (MODELOS[m], TECHO) for m in CON_DATOS},
}
HORIZONTE, FINAL_DIAS = 72, 14               # horizonte comun y la medida "final" (dias 58 a 71)
SALIDA = PROJECT_ROOT / "results" / "correct_base_06_menos_datos"

## 1. Las simulaciones

`run_tasks` simula cada corrida y guarda la caminata de cada batch y el estado del deposito en cada paso
en `results/correct_base_06_menos_datos`; si una corrida ya esta guardada, no la repite. La primera vez
tarda unas dos horas, casi todo `big`, porque los tests largos tardan mas. Las configuraciones nuevas se
agregan al registro de experimentos.

In [ ]:
tareas, donde = [], {}                       # donde[deposito, train, corrida] = (tarea, evaluacion)
for nombre in DEPOSITOS:
    for train in TRAIN:
        instancia = InstanceSpec(nombre, test_size=round(1 - train, 2))
        por_plan = {}                        # un plan se arma una vez y se evalua de una o mas formas
        for corrida, (plan, ev) in CORRIDAS.items():
            por_plan.setdefault(plan, []).append((corrida, ev))
        for plan, evaluaciones in por_plan.items():
            tarea = Task("menos datos", instancia, plan, tuple(ev for _, ev in evaluaciones))
            tareas.append(tarea)
            for i, (corrida, _) in enumerate(evaluaciones):
                donde[nombre, train, corrida] = (tarea.key, i)

t0 = time.time()
estado = run_tasks(tareas, SALIDA, n_jobs={"medium": 2, "big": 1},
                   expected_seconds=lambda t: t.instance.test_size)   # los tests largos primero
print(f"{len(tareas)} tareas; simuladas ahora: {estado['status'].value_counts().to_dict()} "
      f"({(time.time() - t0) / 60:.1f} minutos)")
print("configuraciones nuevas en el registro:", register_results(SALIDA))

In [ ]:
res = load_results(SALIDA)


def leer(carpeta, nombre, train, corrida):
    # Registro por batch ("batches") o por paso ("pasos") de una corrida.
    key, ev = donde[nombre, train, corrida]
    return pd.read_parquet(SALIDA / carpeta / f"{key}.parquet").query("eval == @ev")


def cargar(nombre, train):
    # Caminata de cada batch de todas las corridas de un corte, con su dia y semana desde el corte.
    logs = {c: leer("batches", nombre, train, c) for c in CORRIDAS}
    inicio = logs["hoy"]["timestamp"].min()
    filas = []
    for c, l in logs.items():
        dia = ((l["timestamp"] - inicio).dt.total_seconds() // 86400).astype(int)
        filas.append(pd.DataFrame({"corrida": c, "dia": dia, "caminata": l["distance"].to_numpy()}))
    t = pd.concat(filas, ignore_index=True)
    t["semana"] = t["dia"] // 7
    return t


batches = pd.concat({(n, tr): cargar(n, tr) for n in DEPOSITOS for tr in TRAIN},
                    names=["deposito", "train"]).reset_index(level=[0, 1]).reset_index(drop=True)
largo = batches.groupby(["deposito", "train"])["dia"].max() + 1
pd.DataFrame({"dias de test": largo, "semanas completas": largo // 7,
              "batches de test por corrida": batches.query("corrida == 'hoy'").groupby(["deposito", "train"]).size()})

## 2. Semana a semana desde el corte

Cada panel es un plan; cada linea, un corte (mas clara = menos historia). La linea gris es hoy en el
corte de 10%. Si un plan arranca en frio, sus primeras semanas con poca historia quedan por encima de las
otras; si converge, las lineas se juntan con el tiempo.

In [ ]:
TINTA, TINTA_2, MUDO, LINEA, EJE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
RAMPA = dict(zip(TRAIN, ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]))   # menos historia = mas claro
COLOR = {"aleatorio": "#eb6834", "greedy": "#1baf7a", "aisle_fill": "#eda100", "ganador": "#2a78d6"}
miles = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", "."))


def ejes(ax):
    # Grilla horizontal tenue, sin marco arriba ni a la derecha.
    ax.grid(True, axis="y", color=LINEA, lw=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(EJE)
    ax.tick_params(colors=TINTA_2, labelsize=9)


def semanal(nombre):
    # M_s de cada corrida y corte, solo semanas completas.
    b = batches.query("deposito == @nombre")
    dias = b.query("corrida == 'hoy'").groupby(["train", "semana"])["dia"].nunique()
    m = b.groupby(["train", "semana", "corrida"])["caminata"].mean().unstack("corrida")
    return m.loc[dias[dias == 7].index]


semanas = {n: semanal(n) for n in DEPOSITOS}


def figura_semanas(nombre):
    m = semanas[nombre]
    fig, ejes_m = plt.subplots(1, len(MODELOS), figsize=(13, 3.9), sharey=True)
    hoy = m.loc[TRAIN[0], "hoy"]
    for ax, modelo in zip(ejes_m, MODELOS):
        ax.plot(hoy.index, hoy, color=MUDO, lw=1.2)
        for train in TRAIN:
            serie = m.loc[train, modelo]
            ax.plot(serie.index, serie, color=RAMPA[train], lw=2, label=f"train {train:.0%}")
        ejes(ax)
        ax.set_title(modelo, loc="left", fontsize=10, color=TINTA)
        ax.set_xlabel("semana desde el corte", color=TINTA_2, fontsize=9)
    ejes_m[0].yaxis.set_major_formatter(miles)
    ejes_m[0].set_ylabel("caminata media por batch (pulgadas)", color=TINTA_2, fontsize=9)
    ejes_m[0].set_ylim(0, 1.05 * hoy.max())
    fig.suptitle(f"{nombre}: caminata semanal desde el corte, un corte por linea (gris: hoy)", x=0.02,
                 ha="left", fontsize=11.5, color=TINTA)
    fig.legend(*ejes_m[0].get_legend_handles_labels(), loc="lower center", ncol=len(TRAIN), frameon=False,
               fontsize=9)
    fig.tight_layout(rect=(0, 0.07, 1, 1))
    plt.show()

In [ ]:
figura_semanas("medium")

In [ ]:
figura_semanas("big")

La misma informacion como ventaja sobre el aleatorio del mismo corte, que descuenta lo que comparten todos
los planes (las ordenes de cada semana y la regla). Primeras semanas y ultimas del horizonte comun.

In [ ]:
def ventaja(nombre):
    m = semanas[nombre]
    v = pd.DataFrame({p: 100 * (1 - m[p] / m["aleatorio"]) for p in CON_DATOS})
    return v.unstack("train").loc[[0, 1, 2, 4, 9]].T.round(1)


pd.concat({n: ventaja(n) for n in DEPOSITOS}, names=["deposito", "plan", "train"]).rename(
    columns=lambda s: f"semana {s}")

## 3. Cuanto empeora con menos datos

En el horizonte comun (los primeros 72 dias de cada test), la ventaja sobre el aleatorio del mismo corte
de cada plan: simulado desde el deposito real (promedio de los 72 dias y final, dias 58 a 71) y aplicado
entero (techo, promedio de los 72 dias).

In [ ]:
def horizonte(nombre):
    b = batches.query("deposito == @nombre and dia < @HORIZONTE")
    prom = b.groupby(["train", "corrida"])["caminata"].mean().unstack()
    fin = b.query("dia >= @HORIZONTE - @FINAL_DIAS").groupby(["train", "corrida"])["caminata"].mean().unstack()
    filas = {}
    for p in CON_DATOS:
        filas[p, "promedio"] = 100 * (1 - prom[p] / prom["aleatorio"])
        filas[p, "final"] = 100 * (1 - fin[p] / fin["aleatorio"])
        filas[p, "techo"] = 100 * (1 - prom[f"techo {p}"] / prom["aleatorio"])
    return pd.DataFrame(filas).T


comun = pd.concat({n: horizonte(n) for n in DEPOSITOS}, names=["deposito", "plan", "medida"])
comun.columns = [f"train {t:.0%}" for t in comun.columns]
comun.round(1)

In [ ]:
fig, ejes_d = plt.subplots(1, len(DEPOSITOS), figsize=(11, 4.2), sharey=True)
x = [100 * t for t in TRAIN]
for ax, nombre in zip(ejes_d, DEPOSITOS):
    for p in CON_DATOS:
        ax.plot(x, comun.loc[(nombre, p, "final")], color=COLOR[p], lw=2, marker="o", ms=5,
                label=f"{p}: desde el deposito real (final)")
        ax.plot(x, comun.loc[(nombre, p, "techo")], color=COLOR[p], lw=1.5, ls="--", label=f"{p}: entero (techo)")
    ax.axhline(0, color=MUDO, lw=1)
    ejes(ax)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{t:.0f}%" for t in x])
    ax.set_title(nombre, loc="left", fontsize=10, color=TINTA)
    ax.set_xlabel("historia antes del corte (train)", color=TINTA_2, fontsize=9)
ejes_d[0].set_ylabel("ventaja sobre el aleatorio (%)", color=TINTA_2, fontsize=9)
fig.suptitle("Ventaja de cada plan sobre el aleatorio del mismo corte, primeros 72 dias", x=0.02, ha="left",
             fontsize=11.5, color=TINTA)
fig.legend(*ejes_d[0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.12, 1, 1))
plt.show()

## 4. Se estabiliza con mas tiempo?

Los cortes tempranos tienen tests mas largos. Para cada corte se aplica la regla del notebook 05 a sus
ultimas cuatro semanas completas: el ganador sigue cambiando si su caminata semanal se movio mas que
$2\sqrt{2}\,\sigma$, con $\sigma$ el desvio de las medias semanales de su techo, que no cambia de estado.
Tambien se da que parte del trecho entre el aleatorio y el techo recorrio en la ultima semana,
$100\,(M^{\text{aleatorio}} - M^{\text{ganador}}) / (M^{\text{aleatorio}} - M^{\text{techo}})$.

In [ ]:
def estabilidad(nombre, plan="ganador"):
    m = semanas[nombre]
    filas = {}
    for train in TRAIN:
        t = m.loc[train]
        sigma = t[f"techo {plan}"].std()
        cambio = t[plan].iloc[-1] - t[plan].iloc[-5]
        umbral = 2 * np.sqrt(2) * sigma
        u = t.iloc[-1]
        filas[train] = {"semanas": len(t), "cambio ultimas 4 semanas": cambio, "umbral": umbral,
                        "veces el umbral": abs(cambio) / umbral,
                        "trecho recorrido, ultima semana (%)":
                            100 * (u["aleatorio"] - u[plan]) / (u["aleatorio"] - t[f"techo {plan}"].mean())}
    return pd.DataFrame(filas).T.rename_axis("train")


pd.concat({n: estabilidad(n) for n in DEPOSITOS}).round(2)

## 5. Lectura